# Data Leakage Analysis

During the transition of this project from an experimental notebook-based workflow to a production-oriented architecture, the feature engineering pipeline was redesigned to improve separation for responsibilities and support inference through the FastAPI service.

For this purpose, the original `FeatureBuilder` class was replaced with new `InferenceFeatureBuilder`. The main architectural difference is the introduction of explicit `fit()` and `transform()` methods. This allows preprocessing parameters to be learned during model training and subsequently reused when new data is received by FastAPI service, without fitting the preprocessing steps again during inference.

After this architectural change was implemented in `model.py`, the resulting model showed worse performance compared with the models previously trained. 

The previous model metrics are included in this notebook for comparison.

The difference raised the possibility that the original `FeatureBuilder` implementation introduced **data leakage** during feature engineering. In particular, some preprocessing operations may have been calculated using the entire dataset before the *train/test* split, allowing information from the test set to influence the resulting features.

Therefore, the goal of this notebook is to investigate this hypothesis and determine whether the change in model performance can be explained by data leakage in the original `FeatureBuilder` pipeline.

Model metrics (`model.py`), possible clean results without data leakage problem

Test MAE: 49.80 | Train MAE: 45.23

Test RMSE: 114.08 | Train RMSE: 101.10

Test R2 Score: 0.71 | Train R2 Score: 0.76


In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
import sys
from pathlib import Path
import seaborn as sns
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 500)

sys.path.append(str(Path().cwd().parent.resolve()))

import preprocessing.features as features

builder = features.FeatureBuilder()

df = pd.read_csv(features.DATASET_PATH)

df.shape

/home/carl/notebooks/airbnb_prices_prediction/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


(74111, 29)

## XGBoost with the Original FeatureBuilder

In [2]:
from sklearn.model_selection import train_test_split

df_leakage = df.copy()

df_leakage = builder.get_df(df, use_amenities=True, use_embeddings=True, embedding_pca_components=370)

X = df_leakage.drop(columns=['log_price'])
y = df_leakage['log_price']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    random_state=42,
    test_size=0.2
)

X_train.shape, X_test.shape

Loading embeddings from Parquet /home/carl/notebooks/airbnb_prices_prediction/dataset/description-embeddings.parquet...
Applying PCA (370 components)...
Embeddings shape is: (74111, 370)


((59288, 513), (14823, 513))

In [3]:
from preprocessing.tree_preprocessor import create_tree_preprocessor


cat_features = X_train.select_dtypes(include=['string', 'object']).columns
tree_preprocessor = create_tree_preprocessor(cat_features)

X_train_processed = tree_preprocessor.fit_transform(X_train, y_train)
X_test_processed = tree_preprocessor.transform(X_test)

X_train_processed = X_train_processed.astype(np.float32)
X_test_processed = X_test_processed.astype(np.float32)

X_train_processed.shape, X_test_processed.shape

((59288, 1499), (14823, 1499))

In [4]:
import xgboost as xgb

dtrain = xgb.DMatrix(X_train_processed, label=y_train)
DEVICE = 'cuda' if xgb.build_info()['USE_CUDA'] else 'cpu'
print(DEVICE)

params = {
    "objective": "reg:squarederror",
    "eval_metric": "rmse",
    "tree_method": "hist",
    "device": DEVICE,
    
    "learning_rate": 0.05,
    "max_depth": 6,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    
    "gamma": 0,
    "min_child_weight": 10,
    "reg_alpha": 0.0,
    "reg_lambda": 1.0,
    
    "seed": 42,
    "verbosity": 0
}

cv_results = xgb.cv(
    params=params,
    dtrain=dtrain,
    num_boost_round=1000,
    nfold=3,
    metrics="rmse",
    early_stopping_rounds=50,
    seed=42,
    shuffle=True,
    verbose_eval=False
)

best_num_boost_rounds = len(cv_results)
best_cv_rmse = cv_results["test-rmse-mean"].iloc[-1]

best_num_boost_rounds, best_cv_rmse

cuda


(1000, np.float64(0.37788698171530544))

In [5]:
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

model = xgb.train(
    params=params,
    dtrain=dtrain,
    num_boost_round=best_num_boost_rounds
)

y_pred_test_log = model.predict(xgb.DMatrix(X_test_processed))
y_pred_train_log = model.predict(xgb.DMatrix(X_train_processed))

y_pred_test = np.exp(y_pred_test_log)
y_pred_train = np.exp(y_pred_train_log)

test_MAE = mean_absolute_error(np.exp(y_test), y_pred_test)
train_MAE = mean_absolute_error(np.exp(y_train), y_pred_train)

test_RMSE = root_mean_squared_error(np.exp(y_test), y_pred_test)
train_RMSE = root_mean_squared_error(np.exp(y_train), y_pred_train)

test_r2 = r2_score(y_test, y_pred_test_log)
train_r2 = r2_score(y_train, y_pred_train_log)

print(f"Test MAE: {test_MAE:.2f}$ | Train MAE: {train_MAE:.2f}$")
print(f"Test RMSE: {test_RMSE:.2f}$ | Train RMSE: {train_RMSE:.2f}$")
print(f"Test R2 Score: {test_r2:.2f} | Train R2 Score: {train_r2:.2f}")

Test MAE: 47.94$ | Train MAE: 31.90$
Test RMSE: 110.23$ | Train RMSE: 71.05$
Test R2 Score: 0.73 | Train R2 Score: 0.88


## XGBoost with the Leakage-Free InferenceFeatureBuilder

In [6]:
from preprocessing.inference_features import InferenceFeatureBuilder

df_copy = df.copy()

inference_builder = InferenceFeatureBuilder(
    use_amenities=True,
    use_embeddings=True,
    embedding_pca_components=370
)

X = df_copy.drop(columns=['log_price'])
y = df_copy['log_price']

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y,
    random_state=42,
    test_size=0.2
)

X_train_raw.shape, X_test_raw.shape

Loading weights: 100%|███████████████████████████████| 391/391 [00:00<00:00, 73832.74it/s]


((59288, 28), (14823, 28))

In [7]:
inference_builder.fit(X_train_raw)

X_train = inference_builder.transform(X_train_raw)
X_test = inference_builder.transform(X_test_raw)

X_train.shape, X_test.shape

Loading embedding parquet file from: /home/carl/notebooks/airbnb_prices_prediction/dataset/description-embeddings_01.parquet
Loading embedding parquet file from: /home/carl/notebooks/airbnb_prices_prediction/dataset/description-embeddings_01.parquet
Loading embedding parquet file from: /home/carl/notebooks/airbnb_prices_prediction/dataset/description-embeddings_01.parquet


((59288, 513), (14823, 513))

In [8]:
cat_features = X_train.select_dtypes(include=['string', 'object']).columns

tree_preprocessor = create_tree_preprocessor(cat_features)

X_train_processed = tree_preprocessor.fit_transform(X_train, y_train)
X_test_processed = tree_preprocessor.transform(X_test)

X_train_processed = X_train_processed.astype(np.float32)
X_test_processed = X_test_processed.astype(np.float32)

X_train_processed.shape, X_test_processed.shape

((59288, 1499), (14823, 1499))

In [9]:
dtrain = xgb.DMatrix(X_train_processed, label=y_train)

cv_results_inference = xgb.cv(
    params=params,
    dtrain=dtrain,
    num_boost_round=1000,
    nfold=3,
    metrics="rmse",
    early_stopping_rounds=50,
    seed=42,
    shuffle=True,
    verbose_eval=False
)

best_num_boost_round_infeerence = len(cv_results_inference)

best_cv_results_inference = cv_results_inference["test-rmse-mean"].iloc[-1]

best_cv_results_inference, best_num_boost_round_infeerence

(np.float64(0.39077999250298356), 764)

In [10]:
model = xgb.train(
    params=params,
    dtrain=dtrain,
    num_boost_round=best_num_boost_round_infeerence
)

y_pred_test_log = model.predict(xgb.DMatrix(X_test_processed))
y_pred_train_log = model.predict(xgb.DMatrix(X_train_processed))

y_pred_test = np.exp(y_pred_test_log)
y_pred_train = np.exp(y_pred_train_log)

test_MAE = mean_absolute_error(np.exp(y_test), y_pred_test)
train_MAE = mean_absolute_error(np.exp(y_train), y_pred_train)

test_RMSE = root_mean_squared_error(np.exp(y_test), y_pred_test)
train_RMSE = root_mean_squared_error(np.exp(y_train), y_pred_train)

test_r2 = r2_score(y_test, y_pred_test_log)
train_r2 = r2_score(y_train, y_pred_train_log)

print(f"Test MAE: {test_MAE:.2f}$ | Train MAE: {train_MAE:.2f}$")
print(f"Test RMSE: {test_RMSE:.2f}$ | Train RMSE: {train_RMSE:.2f}$")
print(f"Test R2 Score: {test_r2:.2f} | Train R2 Score: {train_r2:.2f}")

Test MAE: 49.93$ | Train MAE: 39.04$
Test RMSE: 114.24$ | Train RMSE: 88.82$
Test R2 Score: 0.71 | Train R2 Score: 0.82


# Conclusion

The experiments confirmed our hypothesis that the original `FeatureBuilder` introduced data leakage into the machine learning pipeline.

The original `FeatureBuilder` performed several data-dependent preprocessing operations before the train/test split, including PCA fitting, calculation of numerical statistics, identification of frequent neighbourhoods and property types, and calculation of city-level listing centers. As a result, information from the test set could influence the features used to train the model.

This resulted in noticeably better reported performance for the original pipeline:

- Test RMSE: $110.23
- Test R2_score: 0.73

After replacing the original `FeatureBuilder` with the new `InferenceFeatureBuilder`, all data-dependent preprocessing steps were moved into a `fit()` stage and applied to new data through `transform()`. The preprocessing was therefore fitted exclusively on the training data.

The leakage-free pipeline produced:

- Test RMSE: $114.24
- Test R2_score: 0.71

The difference in performance confirms that the original results were overly optimistic and should not be used as the final model evaluation.

Although the leakage-free pipeline currently performs worse, the new architecture provides a more reliable foundation for the final production model. The next step is therefore to re-optimize XGBoost using the corrected pipeline and determine whether the performance can be improved without introducing data leakage.

# Re-optimize the XGBoost model

In [11]:
print(f"Using: {DEVICE}")

def objective(trial):

    params = {
        "objective": "reg:squarederror",
        "eval_metric": "rmse",
        "tree_method": "hist",
        "device": DEVICE,
        
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.10, log=True),
        "max_depth": trial.suggest_int("max_depth", 2, 8),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.4, 1.0),
        "gamma": trial.suggest_float("gamma", 0.0, 5.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 30),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-4, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.1, 30.0, log=True),

        "verbosity": 0,
        "n_jobs": -1,
        "seed": 42
    }

    cv_results = xgb.cv(
        params=params,
        dtrain=dtrain,
        early_stopping_rounds=50,
        num_boost_round=1000,
        nfold=3,
        metrics="rmse",
        seed=42,
        verbose_eval=False,
        shuffle=True,
        callbacks=[XGBoostPruningCallback(trial, "test-rmse")]
    )

    trial.set_user_attr(
        "best_num_boost_round",
        len(cv_results)
    )

    test_rmse = cv_results['test-rmse-mean'].iloc[-1]
    # train_rmse = cv_results['train-rmse-mean'].iloc[-1]

    # gap = (test_rmse - train_rmse) / test_rmse
    
    # trial.set_user_attr("gap", gap)
    # trial.set_user_attr("train_rmse", train_rmse)
    # trial.set_user_attr("test_rmse", test_rmse)

    # score = test_rmse + alpha * gap

    return test_rmse

Using: cuda


In [12]:
df_copy = df.copy()

X = df_copy.drop(columns=['log_price'])
y = df_copy['log_price']

X.shape, y.shape

((74111, 28), (74111,))

In [13]:
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y,
    random_state=42,
    test_size=0.2
)

X_train_raw.shape, X_test_raw.shape

((59288, 28), (14823, 28))

In [14]:
inference_builder = InferenceFeatureBuilder(
    use_amenities=True,
    use_embeddings=True,
    embedding_pca_components=370
)

inference_builder.fit(X_train_raw)

X_train = inference_builder.transform(X_train_raw)
X_test = inference_builder.transform(X_test_raw)

X_train.shape, X_test.shape

Loading weights: 100%|███████████████████████████████| 391/391 [00:00<00:00, 63017.71it/s]


Loading embedding parquet file from: /home/carl/notebooks/airbnb_prices_prediction/dataset/description-embeddings_01.parquet
Loading embedding parquet file from: /home/carl/notebooks/airbnb_prices_prediction/dataset/description-embeddings_01.parquet
Loading embedding parquet file from: /home/carl/notebooks/airbnb_prices_prediction/dataset/description-embeddings_01.parquet


((59288, 513), (14823, 513))

In [15]:
cat_features = X_train.select_dtypes(include=['string', 'object']).columns

tree_preprocessor = create_tree_preprocessor(cat_features)

X_train_processed = tree_preprocessor.fit_transform(X_train, y_train)
X_test_processed = tree_preprocessor.transform(X_test)

X_train_processed = X_train_processed.astype(np.float32)
X_test_processed = X_test_processed.astype(np.float32)

X_train_processed.shape, X_test_processed.shape

((59288, 1499), (14823, 1499))

In [16]:
dtrain = xgb.DMatrix(X_train_processed, label=y_train)

In [17]:
from pathlib import Path

XGBOOST_DIR = Path('../artifacts/xgboost')
XGBOOST_DIR.mkdir(exist_ok=True, parents=True)

XGBOOST_FREE_LEAKAGE = XGBOOST_DIR / "xgboost_free_leakage"

In [18]:
%%time
from utils.inference_xgb_pipeline import InferenceXGBoostPipeline
import optuna
from optuna_integration import XGBoostPruningCallback

model_exists = (
    XGBOOST_FREE_LEAKAGE.with_suffix(".json").exists() and
    XGBOOST_FREE_LEAKAGE.with_suffix(".params").exists() and 
    XGBOOST_FREE_LEAKAGE.with_suffix(".joblib").exists() and
    Path(XGBOOST_FREE_LEAKAGE.parent / f"{XGBOOST_FREE_LEAKAGE.name}_feature_builder.joblib").exists()
)

if model_exists:
    print("Loading existing model...")
    xgboost_pipeline = InferenceXGBoostPipeline.load(XGBOOST_FREE_LEAKAGE)
else:
    print("Training model...")
    optuna.logging.set_verbosity(optuna.logging.INFO)

    study = optuna.create_study(
        direction="minimize",
        pruner=optuna.pruners.MedianPruner(
            n_startup_trials=10, n_warmup_steps=50, interval_steps=10
        )
    )

    study.optimize(objective, n_trials=250, gc_after_trial=True)
    best_num_boost_round = study.best_trial.user_attrs['best_num_boost_round']

    best_params = {
        **study.best_params,
        "objective": "reg:squarederror",
        "eval_metric": "rmse",
        "tree_method": "hist",
        "n_jobs": -1,
        "seed": 42,
        "verbosity": 0,
        "device": DEVICE
    }

    xgboost_pipeline = InferenceXGBoostPipeline(
        preprocessor=tree_preprocessor,
        feature_builder=inference_builder
    )

    xgboost_pipeline.fit(
        X_train, y_train,
        params=best_params,
        num_boost_round=best_num_boost_round
    )

    xgboost_pipeline.save(XGBOOST_FREE_LEAKAGE)

Loading existing model...
CPU times: user 1.36 s, sys: 1.38 s, total: 2.74 s
Wall time: 2.68 s


In [19]:
xgboost_pipeline.params

{'learning_rate': 0.0579461944070209,
 'max_depth': 7,
 'subsample': 0.9795327709843625,
 'colsample_bytree': 0.8160810616223607,
 'gamma': 0.37541438242092273,
 'min_child_weight': 15,
 'reg_alpha': 0.0955993596096246,
 'reg_lambda': 0.12591176359957637,
 'objective': 'reg:squarederror',
 'eval_metric': 'rmse',
 'tree_method': 'hist',
 'n_jobs': -1,
 'seed': 42,
 'verbosity': 0,
 'device': 'cuda'}

In [20]:
y_pred_test_log = xgboost_pipeline.predict(X_test_raw)
y_pred_train_log = xgboost_pipeline.predict(X_train_raw)

y_pred_test = np.exp(y_pred_test_log)
y_pred_train = np.exp(y_pred_train_log)

test_MAE = mean_absolute_error(np.exp(y_test), y_pred_test)
train_MAE = mean_absolute_error(np.exp(y_train), y_pred_train)

test_RMSE = root_mean_squared_error(np.exp(y_test), y_pred_test)
train_RMSE = root_mean_squared_error(np.exp(y_train), y_pred_train)

test_r2 = r2_score(y_test, y_pred_test_log)
train_r2 = r2_score(y_train, y_pred_train_log)

print(f"Test MAE: {test_MAE:.2f}$ | Train MAE: {train_MAE:.2f}$")
print(f"Test RMSE: {test_RMSE:.2f}$ | Train RMSE: {train_RMSE:.2f}$")
print(f"Test R2 Score: {test_r2:.2f} | Train R2 Score: {train_r2:.2f}")

Loading embedding parquet file from: /home/carl/notebooks/airbnb_prices_prediction/dataset/description-embeddings_01.parquet
Loading embedding parquet file from: /home/carl/notebooks/airbnb_prices_prediction/dataset/description-embeddings_01.parquet
Test MAE: 49.80$ | Train MAE: 36.04$
Test RMSE: 114.00$ | Train RMSE: 83.05$
Test R2 Score: 0.71 | Train R2 Score: 0.84


# Overfitting Reduction

In [21]:
print(f"Using: {DEVICE}")

def objective(trial, alpha=0.0):

    params = {
        "objective": "reg:squarederror",
        "eval_metric": "rmse",
        "tree_method": "hist",
        "device": DEVICE,
        
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.10, log=True),
        "max_depth": trial.suggest_int("max_depth", 2, 8),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.4, 1.0),
        "gamma": trial.suggest_float("gamma", 0.0, 5.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 30),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-4, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.1, 30.0, log=True),

        "verbosity": 0,
        "n_jobs": -1,
        "seed": 42
    }

    cv_results = xgb.cv(
        params=params,
        dtrain=dtrain,
        early_stopping_rounds=50,
        num_boost_round=1000,
        nfold=3,
        metrics="rmse",
        seed=42,
        verbose_eval=False,
        shuffle=True,
        callbacks=[XGBoostPruningCallback(trial, "test-rmse")]
    )

    trial.set_user_attr(
        "best_num_boost_round",
        len(cv_results)
    )

    test_rmse = cv_results['test-rmse-mean'].iloc[-1]
    train_rmse = cv_results['train-rmse-mean'].iloc[-1]

    gap = (test_rmse - train_rmse) / test_rmse
    
    trial.set_user_attr("gap", gap)
    trial.set_user_attr("train_rmse", train_rmse)
    trial.set_user_attr("test_rmse", test_rmse)

    score = test_rmse + alpha * gap

    return score

Using: cuda


In [22]:
df_copy = df.copy()

X = df_copy.drop(columns=['log_price'])
y = df_copy['log_price']

X.shape, y.shape

((74111, 28), (74111,))

In [23]:
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y,
    random_state=42,
    test_size=0.2
)

X_train_raw.shape, X_test_raw.shape

((59288, 28), (14823, 28))

In [24]:
inference_builder = InferenceFeatureBuilder(
    use_amenities=True,
    use_embeddings=True,
    embedding_pca_components=370
)

inference_builder.fit(X_train_raw)

X_train = inference_builder.transform(X_train_raw)
X_test = inference_builder.transform(X_test_raw)

X_train.shape, X_test.shape

Loading weights: 100%|███████████████████████████████| 391/391 [00:00<00:00, 65093.79it/s]


Loading embedding parquet file from: /home/carl/notebooks/airbnb_prices_prediction/dataset/description-embeddings_01.parquet
Loading embedding parquet file from: /home/carl/notebooks/airbnb_prices_prediction/dataset/description-embeddings_01.parquet
Loading embedding parquet file from: /home/carl/notebooks/airbnb_prices_prediction/dataset/description-embeddings_01.parquet


((59288, 513), (14823, 513))

In [25]:
cat_features = X_train.select_dtypes(include=['string', 'object']).columns

tree_preprocessor = create_tree_preprocessor(cat_features)

X_train_processed = tree_preprocessor.fit_transform(X_train, y_train)
X_test_processed = tree_preprocessor.transform(X_test)

X_train_processed = X_train_processed.astype(np.float32)
X_test_processed = X_test_processed.astype(np.float32)

X_train_processed.shape, X_test_processed.shape

((59288, 1499), (14823, 1499))

In [26]:
dtrain = xgb.DMatrix(X_train_processed, label=y_train)

In [27]:
STUDY_DIR = Path('../artifacts/xgboost/study')
STUDY_DIR.mkdir(exist_ok=True, parents=True)

In [28]:
%%time
import joblib

alphas = [0.01, 0.03, 0.05, 0.07, 0.10]

summary = []

optuna.logging.set_verbosity(optuna.logging.WARNING)

for alpha in alphas:

    print("=" * 70)
    print(f"Alpha = {alpha}")
    print("=" * 70)

    study_path = STUDY_DIR / f"study_alpha_01_{alpha:.2f}.joblib"
    trials_path = STUDY_DIR / f"study_alpha_01_{alpha:.2f}.parquet"

    if study_path.exists():

        print("Loading study...")
        study = joblib.load(study_path)

        if not trials_path.exists():
            study.trials_dataframe(
                attrs=("number", "value", "params", "user_attrs", "state")
            ).to_parquet(trials_path, index=False)

    else:

        print("Training study...")

        study = optuna.create_study(
            direction="minimize",
            pruner=optuna.pruners.MedianPruner(
                n_startup_trials=10,
                n_warmup_steps=50,
                interval_steps=10
            )
        )

        study.optimize(
            lambda trial: objective(trial, alpha=alpha),
            n_trials=100,
            gc_after_trial=True
        )

        joblib.dump(study, study_path)

        study.trials_dataframe(
            attrs=("number", "value", "params", "user_attrs", "state")
        ).to_parquet(trials_path, index=False)

    best_trial = study.best_trial

    summary.append({
        "alpha": alpha,
        "score": best_trial.value,
        "test_rmse": best_trial.user_attrs["test_rmse"],
        "train_rmse": best_trial.user_attrs["train_rmse"],
        "gap": best_trial.user_attrs["gap"],
        "best_num_boost_round": best_trial.user_attrs["best_num_boost_round"],
        **best_trial.params
    })

summary = (
    pd.DataFrame(summary)
    .sort_values("alpha")
    .reset_index(drop=True)
)

summary.to_parquet(
    STUDY_DIR / "summary_01.parquet",
    index=False
)

summary

Alpha = 0.01
Loading study...
Alpha = 0.03
Loading study...
Alpha = 0.05
Loading study...
Alpha = 0.07
Loading study...
Alpha = 0.1
Loading study...
CPU times: user 165 ms, sys: 0 ns, total: 165 ms
Wall time: 164 ms


,alpha,score,test_rmse,train_rmse,gap,best_num_boost_round,learning_rate,max_depth,subsample,colsample_bytree,gamma,min_child_weight,reg_alpha,reg_lambda
0,0.01,0.393921,0.391511,0.297141,0.241040,547,0.076574,6,0.951482,0.760404,0.476302,8,0.172761,14.716133
1,0.03,0.395304,0.392588,0.357056,0.090507,1000,0.067898,4,0.775092,0.807291,1.589848,14,0.000733,5.213648
2,0.05,0.397372,0.392471,0.353998,0.098027,688,0.066727,4,0.703662,0.945879,1.452733,27,0.000445,0.967605
3,0.07,0.398477,0.393805,0.367523,0.066739,949,0.089988,4,0.855035,0.489703,1.816667,20,0.025410,0.185641
4,0.10,0.400131,0.394202,0.370832,0.059285,1000,0.091474,4,0.936316,0.729115,1.826059,26,0.038062,0.121613


In [29]:
study_path = STUDY_DIR / "study_alpha_01_0.03.joblib"

study = joblib.load(study_path)

best_trial = study.best_trial

print("Best params:")
print(best_trial.params)

print("\nBest score:", best_trial.value)
print("Test RMSE:", best_trial.user_attrs["test_rmse"])
print("Train RMSE:", best_trial.user_attrs["train_rmse"])
print("Gap:", best_trial.user_attrs["gap"])
print("Boost rounds:", best_trial.user_attrs["best_num_boost_round"])

Best params:
{'learning_rate': 0.067898198431014, 'max_depth': 4, 'subsample': 0.775092042172847, 'colsample_bytree': 0.8072914452007689, 'gamma': 1.5898476193887499, 'min_child_weight': 14, 'reg_alpha': 0.0007326551645415868, 'reg_lambda': 5.21364776002228}

Best score: 0.3953036316719079
Test RMSE: 0.3925884265869907
Train RMSE: 0.3570564901820366
Gap: 0.09050683616390524
Boost rounds: 1000


In [30]:
best_params = {
    **best_trial.params,
    "objective": "reg:squarederror",
    "eval_metric": "rmse",
    "tree_method": "hist",
    "n_jobs": -1,
    "seed": 42,
    "verbosity": 0,
    "device": DEVICE
}

In [31]:
xgboost_pipeline_alpha_03 = InferenceXGBoostPipeline(
    preprocessor=tree_preprocessor,
    feature_builder=inference_builder
)

In [32]:
xgboost_pipeline_alpha_03.fit(
    X_train,
    y_train,
    params=best_params,
    num_boost_round=best_trial.user_attrs["best_num_boost_round"]
)

In [33]:
y_pred_test_log = xgboost_pipeline_alpha_03.predict(X_test_raw)
y_pred_train_log = xgboost_pipeline_alpha_03.predict(X_train_raw)

y_pred_test = np.exp(y_pred_test_log)
y_pred_train = np.exp(y_pred_train_log)

test_MAE = mean_absolute_error(np.exp(y_test), y_pred_test)
train_MAE = mean_absolute_error(np.exp(y_train), y_pred_train)

test_RMSE = root_mean_squared_error(np.exp(y_test), y_pred_test)
train_RMSE = root_mean_squared_error(np.exp(y_train), y_pred_train)

test_r2 = r2_score(y_test, y_pred_test_log)
train_r2 = r2_score(y_train, y_pred_train_log)

print(f"Test MAE: {test_MAE:.2f}$ | Train MAE: {train_MAE:.2f}$")
print(f"Test RMSE: {test_RMSE:.2f}$ | Train RMSE: {train_RMSE:.2f}$")
print(f"Test R2 Score: {test_r2:.2f} | Train R2 Score: {train_r2:.2f}")

Loading embedding parquet file from: /home/carl/notebooks/airbnb_prices_prediction/dataset/description-embeddings_01.parquet
Loading embedding parquet file from: /home/carl/notebooks/airbnb_prices_prediction/dataset/description-embeddings_01.parquet
Test MAE: 50.01$ | Train MAE: 45.92$
Test RMSE: 114.44$ | Train RMSE: 103.10$
Test R2 Score: 0.71 | Train R2 Score: 0.75


# XGBoost Experiments Conclusion

The conducted experiments showed the relative differences in performance between the different XGBoost configurations. Altough data leakage was identified in the `FeatureBuilder` during development, it was present in the common data preparation pipeline used across these experiments. Therefore, the absoulte metrics on these models cannot be considered as a final evaluation of their performance. However, the results can still be used to compare the models relatives to each other and identify the most promissing configuration.

For this reason, there is no need to re-evaluate every previous XGBoost configuration without data leakage.

# CatBoost Data Leakage Analysis

We will conduct a similar experiment for CatBoost by comparing the same model configuration under two different feature-building approaches:

* **Original** `FeatureBuilder` - with the previously identified data leakage.
* **New** `InferenceFeatureBuilder` - with the data leakage removed.

This will allow us to deetermine how the leakage affects the model's performance.

## CatBoost with the Original FeatureBuilder

In [34]:
from catboost.utils import get_gpu_device_count

DEVICE = "GPU" if  get_gpu_device_count() > 0 else "CPU"
DEVICE

'GPU'

In [35]:
import catboost
from catboost import Pool, CatBoostRegressor

df_copy = builder.get_df(df, use_amenities=True, use_embeddings=True, embedding_pca_components=370)

X = df_copy.drop(columns=['log_price'])
y = df_copy['log_price']

X.shape, y.shape

Loading embeddings from Parquet /home/carl/notebooks/airbnb_prices_prediction/dataset/description-embeddings.parquet...
Applying PCA (370 components)...
Embeddings shape is: (74111, 370)


((74111, 513), (74111,))

In [36]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    random_state=42,
    test_size=0.2
)

X_train.shape, X_test.shape

((59288, 513), (14823, 513))

In [37]:
cat_features = X_train.select_dtypes(include=['string', 'object']).columns.tolist()

train_pool = Pool(X_train, label=y_train, cat_features=cat_features)
test_pool = Pool(X_test, label=y_test, cat_features=cat_features)

train_pool.shape

(59288, 513)

In [38]:
catboost_pipeline = CatBoostRegressor(
    loss_function="RMSE",
    eval_metric="RMSE",
    random_seed=42,
    task_type=DEVICE,
    verbose=False
)

catboost_pipeline.fit(train_pool)

y_pred_test_log = catboost_pipeline.predict(test_pool)
y_pred_train_log = catboost_pipeline.predict(train_pool)

y_pred_test = np.exp(y_pred_test_log)
y_pred_train = np.exp(y_pred_train_log)

test_MAE = mean_absolute_error(np.exp(y_test), y_pred_test)
train_MAE = mean_absolute_error(np.exp(y_train), y_pred_train)

test_RMSE = root_mean_squared_error(np.exp(y_test), y_pred_test)
train_RMSE = root_mean_squared_error(np.exp(y_train), y_pred_train)

test_r2 = r2_score(y_test, y_pred_test_log)
train_r2 = r2_score(y_train, y_pred_train_log)

print(f"Test MAE: {test_MAE:.2f}$ | Train MAE: {train_MAE:.2f}$")
print(f"Test RMSE: {test_RMSE:.2f}$ | Train RMSE: {train_RMSE:.2f}$")
print(f"Test R2 Score: {test_r2:.2f} | Train R2 Score: {train_r2:.2f}")

Test MAE: 48.46$ | Train MAE: 42.89$
Test RMSE: 111.35$ | Train RMSE: 94.74$
Test R2 Score: 0.73 | Train R2 Score: 0.79


## CatBoost with the Leakage-Free InferenceFeatureBuilder

In [39]:
df_copy = df.copy()

X = df_copy.drop(columns=['log_price'])
y = df_copy['log_price']

X.shape, y.shape

((74111, 28), (74111,))

In [40]:
inference_builder = InferenceFeatureBuilder(
    use_amenities=True,
    use_embeddings=True,
    embedding_pca_components=370
)

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y,
    random_state=42,
    test_size=0.2
)

X_train_raw.shape, X_test_raw.shape

Loading weights: 100%|███████████████████████████████| 391/391 [00:00<00:00, 67978.15it/s]


((59288, 28), (14823, 28))

In [41]:
inference_builder.fit(X_train_raw)

X_train = inference_builder.transform(X_train_raw)
X_test = inference_builder.transform(X_test_raw)

X_train.shape, X_test.shape

Loading embedding parquet file from: /home/carl/notebooks/airbnb_prices_prediction/dataset/description-embeddings_01.parquet
Loading embedding parquet file from: /home/carl/notebooks/airbnb_prices_prediction/dataset/description-embeddings_01.parquet
Loading embedding parquet file from: /home/carl/notebooks/airbnb_prices_prediction/dataset/description-embeddings_01.parquet


((59288, 513), (14823, 513))

In [42]:
cat_features = X_train.select_dtypes(include=['string', 'object']).columns.tolist()

train_pool = Pool(X_train, label=y_train, cat_features=cat_features)
test_pool = Pool(X_test, label=y_test, cat_features=cat_features)

train_pool.shape

(59288, 513)

In [43]:
catboost_pipeline = CatBoostRegressor(
    loss_function="RMSE",
    eval_metric="RMSE",
    random_seed=42,
    task_type=DEVICE,
    verbose=False
)

catboost_pipeline.fit(train_pool)

y_pred_test_log = catboost_pipeline.predict(test_pool)
y_pred_train_log = catboost_pipeline.predict(train_pool)

y_pred_test = np.exp(y_pred_test_log)
y_pred_train = np.exp(y_pred_train_log)

test_MAE = mean_absolute_error(np.exp(y_test), y_pred_test)
train_MAE = mean_absolute_error(np.exp(y_train), y_pred_train)

test_RMSE = root_mean_squared_error(np.exp(y_test), y_pred_test)
train_RMSE = root_mean_squared_error(np.exp(y_train), y_pred_train)

test_r2 = r2_score(y_test, y_pred_test_log)
train_r2 = r2_score(y_train, y_pred_train_log)

print(f"Test MAE: {test_MAE:.2f}$ | Train MAE: {train_MAE:.2f}$")
print(f"Test RMSE: {test_RMSE:.2f}$ | Train RMSE: {train_RMSE:.2f}$")
print(f"Test R2 Score: {test_r2:.2f} | Train R2 Score: {train_r2:.2f}")

Test MAE: 50.03$ | Train MAE: 44.80$
Test RMSE: 114.25$ | Train RMSE: 100.00$
Test R2 Score: 0.71 | Train R2 Score: 0.77


# Conclusion

After removing data leakage, XGBoost and CatBoost achieved the similar performance with an RMSE of approximately 114.00 and 114.25 respectively. The difference of only 0.25 is negligible from a practical perspective.

Therefore, CatBoost was selected as the final model. Altough XGBoost achieved a slightly lower RMSE, CatBoost provides a significantly simpler training workflow, requires no hyperparameter optimization and offers better simplicity and control over the overall pipeline.

Given the nearly identical predictive performance, the simplicity and maintainability of the CatBoost pipeline make it the more practical choice for this project. 